# Train balance bot using PPO with domain randomization

Run each cell by pressing `shift + enter`.

We redo the original training and add more phases to our curriculum learning. In addition to training the bot to stay upright (phase 1) and reduce movement off the origin (phase 2), we introduce increasingly difficult domain randomization schemes in phases 3, 4 and 5.

In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

In [2]:
# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import the custom environment and PPO training module
from envs.balance_bot_env_dr import BalanceBotEnv, DomainRandomConfig
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx
from rl.onnx_actor_to_c import export_onnx_actor_to_c

In [3]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 8               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment

## Configure PPO and environment

In [4]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 32,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 32,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [5]:
def make_balance_bot_env(render, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path    = MJCF_PATH,
        render_mode  = "human" if render else None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), kw=kwargs: make_balance_bot_env(render, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [6]:
def load_agent(envs, model_path):
    """
    For debugging only! Use this to load a previously trained model to skip previous phases. Note 
    that you will still need to run the cells in each prior phase that update the experiment name 
    and environment.
    """
    from rl.ppo_trainer import Agent, TrainResult

    # Make sure model_path is a Path
    model_path = Path(model_path)
    
    # Load agent from previous run
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = Agent(envs, ppo_config).to(device)
    agent.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    agent.eval()
    print(f"Loaded model from {model_path}")
    
    # Wrap agent in a dummy result
    return TrainResult(
        agent = agent,
        checkpoint_dir = model_path.parent,
        best_model_path = model_path,
        final_model_path = None,
        best_mean_return = 0,
    )

## Phase 1: Balance only

We'll start with the easy task of only penalizing if the robot tilts (pitches) forward.

In [7]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-1"

# Create an environment that only rewards staying upright
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    position_penalty_coef=0.0,
    yaw_penalty_coef=0.0
)

In [8]:
# Choo choo train
result = train(ppo_config, envs=envs)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-1__42__1778903990
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-1
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-1__42__1778903990/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=8394.61) to runs/BalanceBot-v0__balance-bot-phase-1__42__1778903990/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-1__42__1778903990/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=9931.00) to runs/BalanceBot-v0__balance-bot-phase-1__42__1778903990/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-1__42__1778903990/checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-1__42__1778903990/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9983.25) to runs/BalanceBot-v0__balance-bot-phase-1__42__1778903990/best_model.cleanrl_model
Final model saved to runs/BalanceBot-v0__b

In [9]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 2045.15
Exported charts_metrics.csv (5 metrics, 1682 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 2: Penalize position and rotation

We start with the agent (actor and critic networks) trained in phase 1. We then update the existing environments to apply a penalty for any motion off the origin as well as rotation around the Z axis (yaw). We then train the existing agent further in this new environment.

In [10]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-2"

# Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.position_penalty_coef = 0.001
    env.yaw_penalty_coef = 0.1

In [11]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-2__42__1778905085
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-2
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-2__42__1778905085/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=9982.48) to runs/BalanceBot-v0__balance-bot-phase-2__42__1778905085/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-2__42__1778905085/checkpoint_iter0100.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-2__42__1778905085/checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=9994.32) to runs/BalanceBot-v0__balance-bot-phase-2__42__1778905085/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-2__42__1778905085/checkpoint_iter0200.cleanrl_model
Final model saved to runs/BalanceBot-v0__balance-bot-phase-2__42__1778905085/balance-bot-phase-2_final.cleanrl_model
Best model mean return: 9994.32, saved to runs/Balan

In [12]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 2047.21
Exported charts_metrics.csv (5 metrics, 293 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 3: Observation noise and action delay

With the robot staying upright and mostly in place, we add the additional difficulty of randomizing the observations (injecting noise) and introducing a random 0..1 timestep delay of how long it takes the chosen action to take effect.

The noisy observations model how real IMUs work. You will often find some noise in their readings, and it can help the robot deal with biases from an imperfectly calibrated IMU.

The action delay models the time it takes for the robot to respond to a chosen action. In simulation, a chosen action takes effect on the very next `step()` call. In real hardware, it could take up to a few milliseconds, thanks to extra code, communication bus latency (e.g. I2C), and motor lag.

In [13]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-3"

# Create data randomization config to add obs noise and action delay
dr = DomainRandomConfig(
    pitch_noise_std_dev=0.01,       # Inject some noise into pitch observation
    pitch_rate_noise_std_dev=0.01,  # Inject some noise into pitch rate observation
    wheel_vel_noise_std_dev = 0.1,  # Inject some noise into wheel velocity observation
    action_delay_steps=1,           # 1 step (5ms) delay
    action_delay_random=True,       # Vary 0-1 steps each episode
)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [14]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot-phase-2__42__1778853328/best_model.cleanrl_model")

In [15]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-3__42__1778906144
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-3
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-3__42__1778906144/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=9978.86) to runs/BalanceBot-v0__balance-bot-phase-3__42__1778906144/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-3__42__1778906144/checkpoint_iter0100.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-3__42__1778906144/checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-3__42__1778906144/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9981.58) to runs/BalanceBot-v0__balance-bot-phase-3__42__1778906144/best_model.cleanrl_model
Final model saved to runs/BalanceBot-v0__balance-bot-phase-3__42__1778906144/balance-bot-phase-3_final.cleanrl_model
Best model mean return: 9981.58, saved to runs/Balan

In [16]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 2042.49
Exported charts_metrics.csv (5 metrics, 466 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 4

In [17]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-4"

# Add motor noise and pushes to our domain randomization config
dr.motor_noise_scale = 0.02 # +/-2% motor noise
dr.push_prob = 0.005        # 0.5% chance of push per step
dr.push_force_max_n = 0.3   # Gentle pushes

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [18]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-4__42__1778907263
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-4
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-4__42__1778907263/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=6098.75) to runs/BalanceBot-v0__balance-bot-phase-4__42__1778907263/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-4__42__1778907263/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=6144.00) to runs/BalanceBot-v0__balance-bot-phase-4__42__1778907263/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-4__42__1778907263/checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-4__42__1778907263/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9983.94) to runs/BalanceBot-v0__balance-bot-phase-4__42__1778907263/best_model.cleanrl_model
Final model saved to runs/BalanceBot-v0__b

In [19]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 2044.08
Exported charts_metrics.csv (5 metrics, 1673 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 5

In [20]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-5"

# Add mass and firction randomization to the DR config
dr.mass_scale_range= (0.8, 1.2)       # +/-20% mass variation
dr.friction_scale_range = (0.5, 1.5)  # 50-150% friction variation

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [21]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-5__42__1778908520
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-5
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-5__42__1778908520/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=2293.46) to runs/BalanceBot-v0__balance-bot-phase-5__42__1778908520/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-5__42__1778908520/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=9458.57) to runs/BalanceBot-v0__balance-bot-phase-5__42__1778908520/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-5__42__1778908520/checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-5__42__1778908520/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9961.75) to runs/BalanceBot-v0__balance-bot-phase-5__42__1778908520/best_model.cleanrl_model
Final model saved to runs/BalanceBot-v0__b

In [22]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 2039.67
Exported charts_metrics.csv (5 metrics, 717 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 6

In [23]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-6"

# Add motor gain randomization to our DR config
dr.motor_gain_range = (0.6, 1.0)    # 60% to 100% motor torque per episode

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [24]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-6__42__1778909777
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-6
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-6__42__1778909777/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=8981.65) to runs/BalanceBot-v0__balance-bot-phase-6__42__1778909777/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-6__42__1778909777/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=9937.57) to runs/BalanceBot-v0__balance-bot-phase-6__42__1778909777/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-6__42__1778909777/checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-6__42__1778909777/checkpoint_iter0200.cleanrl_model
Final model saved to runs/BalanceBot-v0__balance-bot-phase-6__42__1778909777/balance-bot-phase-6_final.cleanrl_model
Best model mean return: 9937.57, saved to runs/Balan

In [25]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 2034.07
Exported charts_metrics.csv (5 metrics, 647 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 7

In [26]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-7"

# Add motor gain randomization to our DR config
dr.ridge_prob=0.05                  # 5% possibility of tire "ridge"
dr.ridge_torque_max_nm=0.005        # Add a slight torque to the joints (simulate tire ridges)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [27]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-7__42__1778911030
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-7
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-7__42__1778911030/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=9966.36) to runs/BalanceBot-v0__balance-bot-phase-7__42__1778911030/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-7__42__1778911030/checkpoint_iter0100.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-7__42__1778911030/checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-7__42__1778911030/checkpoint_iter0200.cleanrl_model
Final model saved to runs/BalanceBot-v0__balance-bot-phase-7__42__1778911030/balance-bot-phase-7_final.cleanrl_model
Best model mean return: 9966.36, saved to runs/BalanceBot-v0__balance-bot-phase-7__42__1778911030/best_model.cleanrl_model
Best model: runs/BalanceBot-v0__balance-bot-phase-7__42_

In [28]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 2041.12
Exported charts_metrics.csv (5 metrics, 293 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Clean up and save model

At this point, we are done with training. We want to delete the environments and save the best actor from our final training phase. We'll export this actor network as an ONNX file that can be used on a variety of hardware platforms.

In [29]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3
Closing env 4
Closing env 5
Closing env 6
Closing env 7


In [30]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

/workspace/software/rl/ppo_trainer.py:381: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0516 13:23:45.943000 2870 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0516 13:23:45.946000 2870 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0516 13:23:45.947000 2870 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs/BalanceBot-v0__balance-bot-phase-7__42__1778911030/actor.onnx


/opt/pyenv/versions/3.12.13/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [31]:
# Export actor to .h file
export_onnx_actor_to_c(
    onnx_path   = result.checkpoint_dir / "actor.onnx",
    output_path = result.checkpoint_dir / "actor.h"
)

Weights found:
  0.weight: shape=(32, 4), dtype=float32
  0.bias: shape=(32,), dtype=float32
  2.weight: shape=(32, 32), dtype=float32
  2.bias: shape=(32,), dtype=float32
  4.weight: shape=(2, 32), dtype=float32
  4.bias: shape=(2,), dtype=float32
C header written to runs/BalanceBot-v0__balance-bot-phase-7__42__1778911030/actor.h
  Layers:  3
  Obs:     4
  Actions: 2


PosixPath('runs/BalanceBot-v0__balance-bot-phase-7__42__1778911030/actor.h')